#### Parameter cell

In [ ]:
# Tagged Parameters cell
cropped_path = "Files/development/cropped/image2_20260501_094512_0.jpg"  # default for testing

#### Imports

In [ ]:
import subprocess
subprocess.run(["pip", "install", "azure-cognitiveservices-vision-customvision", "msrest", "--quiet"])

import importlib, site
importlib.invalidate_caches()
importlib.reload(site)

from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
from msrest.authentication import ApiKeyCredentials
from datetime import datetime
from pyspark.sql import Row
import os

#### Credentials

In [ ]:
vault_url  = "https://cv-training-key.vault.azure.net/"

# Prediction — for running inference
key_p      = notebookutils.credentials.getSecret(vault_url, "cv-prediction-api-key")
endpoint_p = notebookutils.credentials.getSecret(vault_url, "cv-endpoint-p-key")

# Training — READ ONLY, used only to discover available projects dynamically
key_t      = notebookutils.credentials.getSecret(vault_url, "cv-training-api-key")
endpoint_t = notebookutils.credentials.getSecret(vault_url, "cv-endpoint-t-key")

predictor = CustomVisionPredictionClient(
    endpoint_p,
    ApiKeyCredentials(in_headers={"Prediction-key": key_p})
)
trainer = CustomVisionTrainingClient(
    endpoint_t,
    ApiKeyCredentials(in_headers={"Training-key": key_t})
)

#### Get absolute path cell

In [ ]:
files_listing  = notebookutils.fs.ls("Files")
lakehouse_root = files_listing[0].path.split("/Files/")[0]
abs_cropped    = f"{lakehouse_root}/{cropped_path}"

# Copy to /tmp/ for PIL/CV processing
local_tmp = f"/tmp/inference_input.jpg"
notebookutils.fs.cp(abs_cropped, f"file:{local_tmp}")

filename = os.path.basename(cropped_path)
print(f"✅ Image ready for inference: {filename}")

#### Run inference against all models cell

In [ ]:
rows          = []
timestamp     = datetime.now()

# Note: trainer is used READ-ONLY to dynamically discover all published pet-classifier projects at runtime.
# This avoids hardcoding model sizes and scales automatically when new sizes are added to pl_ml_training.
all_projects = {
    p.name: p for p in trainer.get_projects()
    if p.name.startswith("pet-classifier-")
}

print(f"✅ Discovered {len(all_projects)} models: {list(all_projects.keys())}")

for project_name, project in all_projects.items():

    # Extract size from project name e.g. "pet-classifier-128" -> "128"
    size = project_name.replace("pet-classifier-","")
    publish_name = f"publish_{size}"
    

    try:
        with open(local_tmp, "rb") as img:
            results = predictor.classify_image(
                project.id,
                publish_name,
                img.read()
            )

        if not results.predictions:
            print(f"⚠️ No predictions for size {size}")
            continue

        top         = results.predictions[0]
        predicted   = top.tag_name
        confidence  = float(top.probability)

        rows.append(Row(
            image_name  = filename,
            model_size  = size,
            predicted   = predicted,
            confidence  = confidence,
            timestamp   = timestamp
        ))

        print(f"✅ Model {size}: '{predicted}' ({confidence:.2%})")

    except Exception as e:
        print(f"❌ Error on model {size}: {e}")

print(f"\n📦 Inference complete: {len(rows)} model(s) evaluated")

#### Save to inference_metrics cell

In [ ]:
if rows:
    inference_df = spark.createDataFrame(rows)
    inference_df.write.mode("append").saveAsTable("inference_metrics")
    print(f"✅ Saved {len(rows)} inference results for '{filename}'")
else:
    print("⚠️ No results to save")